In [ ]:
df = pd.read_csv("ALL_SAHAM_RELEVANCY_NEWS.csv")
df

In [ ]:
df=df.drop(columns=["Unnamed: 0"])
df.head()

In [ ]:
df.isna().sum()

In [ ]:
!pip install -q transformers torch pandas

import pandas as pd
import re
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

# load model
MODEL_ID = "taufiqdp/indonesian-sentiment"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)

print("Label mapping (id2label):", model.config.id2label)

device = 0 if torch.cuda.is_available() else -1
sentiment_pipe = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=device,
    return_all_scores=False
)

In [ ]:
# batch prediksi
def predict_batch(texts, batch_size=32):
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        preds = sentiment_pipe(batch, truncation=True, max_length=512)
        results.extend(preds)
    return results

In [ ]:
#keyword saham vs override
positif_keywords = [
    "menguat", "rebound", "bullish", "surplus", "naik", "melonjak",
    "penguatan", "stabil", "optimis", "pulih", "rekor", "melambung"
]
negatif_keywords = [
    "melemah", "anjlok", "bearish", "defisit", "turun", "merosot",
    "penurunan", "tertekan", "resesi", "krisis", "minus", "collapse", "pelemahan"
]

positif_pattern = re.compile(r"\b(" + "|".join(positif_keywords) + r")\b", re.IGNORECASE)
negatif_pattern = re.compile(r"\b(" + "|".join(negatif_keywords) + r")\b", re.IGNORECASE)

def apply_keyword_override(text, model_label, model_score):
    pos_match = bool(positif_pattern.search(text))
    neg_match = bool(negatif_pattern.search(text))

    if pos_match and not neg_match:
        return "positif", 1.0
    elif neg_match and not pos_match:
        return "negatif", 1.0
    elif pos_match and neg_match:
        # kalau kedua jenis keyword ada, pilih berdasarkan jumlah kemunculan
        pos_count = len(positif_pattern.findall(text))
        neg_count = len(negatif_pattern.findall(text))
        if pos_count > neg_count:
            return "positif", 1.0
        elif neg_count > pos_count:
            return "negatif", 1.0
        else:
            # seimbang → fallback ke hasil model
            return model_label, model_score
    else:
        # tidak ada keyword → pake hasil model
        return model_label, model_score

In [ ]:
df["konten_clean"] = df["konten_clean"].fillna("").astype(str)

preds = predict_batch(df["konten_clean"].tolist(), batch_size=32)
df["sentiment_label_model"] = [p["label"] for p in preds]
df["sentiment_score_model"] = [float(p["score"]) for p in preds]

# Jika id2label mapping dalam bahasa Inggris, ubah ke bahasa Indonesia:
mapping = {}
for idx, lbl in model.config.id2label.items():
    lbl_low = lbl.lower()
    if "neg" in lbl_low:
        mapping[lbl] = "negatif"
    elif "neu" in lbl_low:
        mapping[lbl] = "netral"
    elif "pos" in lbl_low:
        mapping[lbl] = "positif"
    else:
        mapping[lbl] = lbl  # fallback

# Ubah label model ke versi Indonesia
df["sentiment_label_model_id"] = df["sentiment_label_model"].map(mapping).fillna(df["sentiment_label_model"])

# override keyword
adjusted = [
    apply_keyword_override(text, lbl_id, scr)
    for text, lbl_id, scr in zip(df["konten_clean"],
                                 df["sentiment_label_model_id"],
                                 df["sentiment_score_model"])
]
df["sentiment_label"], df["sentiment_score"] = zip(*adjusted)

In [ ]:
df.to_csv("df_sentiment_saham_taufiqdp.csv", index=False)
print("Selesai — file ditulis: df_sentiment_saham_taufiqdp.csv")
print("Distribusi sentiment_label:", df["sentiment_label"].value_counts())

In [ ]:
import pandas as pd
df = pd.read_csv("df_sentiment_saham_taufiqdp.csv")
df

In [ ]:
from sklearn.utils import resample

df_pos = df[df["sentiment_label"] == "positif"]
df_net = df[df["sentiment_label"] == "netral"]
df_neg = df[df["sentiment_label"] == "negatif"]

TARGET = 4000

# resampling
df_pos_bal = resample(df_pos, replace=False, n_samples=TARGET, random_state=42)  # undersample
df_net_bal = resample(df_net, replace=False, n_samples=TARGET, random_state=42)  # undersample
df_neg_bal = resample(df_neg, replace=True,  n_samples=TARGET, random_state=42)  # oversample

# gabung
df_balanced = pd.concat([df_pos_bal, df_net_bal, df_neg_bal])

# shuffle
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

df_balanced.to_csv("df_sentiment_saham_balanced.csv", index=False)
print("Selesai — data balanced disimpan ke df_sentiment_saham_balanced.csv")
print("Distribusi baru:")
print(df_balanced["sentiment_label"].value_counts())

In [ ]:
df=pd.read_csv("df_sentiment_saham_balanced.csv")
df